In [ ]:
import pandas as pd
import numpy as np



In [14]:
!pip install vaderSentiment

In [22]:
from scipy.stats import zscore
df = pd.read_excel("Data_Train.xlsx")
df['Title len']=df['Title'].str.len()
df['Reviews'] = df['Reviews'].str.extract(r'(\d+)')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Ratings'] = df['Ratings'].str.extract(r'(\d+)')
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = zscore(df['Reviews'])
df['Ratings'] = zscore(df['Ratings'])
df['Rating Review score']=df['Ratings'] * df['Reviews']
df['Title Char length']=df['Title'].str.len()
df['Synopsis len']=df['Synopsis'].str.len()
df['Author freq'] = df['Author'].map(df['Author'].value_counts())

In [23]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get polarity
def get_sentiment(text):
    score = analyzer.polarity_scores(text)
    return score['compound']  # Compound score gives overall sentiment [-1, 1]

# Apply to column
sentiment_score = df['Synopsis'].apply(get_sentiment)

# Optional: Classify into Positive, Negative, Neutral
def classify_sentiment(score):
    if score >= 0.05:
        return 1
    elif score <= -0.05:
        return -1
    else:
        return 0

df['Synopsis Polarity'] = sentiment_score.apply(classify_sentiment)



In [24]:
df.head(20)

,Title,Author,Edition,Reviews,Ratings,Synopsis,Genre,BookCategory,Price,Title len,Rating Review score,Title Char length,Synopsis len,Author freq,Synopsis Polarity
0,The Prisoner's Gold (The Hunters 3),Chris Kuzneski,"Paperback,– 10 Mar 2016",0.050347,-0.259449,THE HUNTERS return in their third brilliant no...,Action & Adventure (Books),Action & Adventure,220.00,35,-0.013062,35,791,4,1
1,Guru Dutt: A Tragedy in Three Acts,Arun Khopkar,"Paperback,– 7 Nov 2012",-1.269032,-0.188133,A layered portrait of a troubled genius for wh...,Cinema & Broadcast (Books),"Biographies, Diaries & True Accounts",202.93,34,0.238747,34,1146,1,1
2,Leviathan (Penguin Classics),Thomas Hobbes,"Paperback,– 25 Feb 1982",0.050347,-0.283221,"""During the time men live without a common Pow...",International Relations,Humour,299.00,28,-0.014259,28,1662,3,1
3,A Pocket Full of Rye (Miss Marple),Agatha Christie,"Paperback,– 5 Oct 2017",0.050347,-0.200019,A handful of grain is found in the pocket of a...,Contemporary Fiction (Books),"Crime, Thriller & Mystery",180.00,34,-0.010070,34,426,69,-1
4,LIFE 70 Years of Extraordinary Photography,Editors of Life,"Hardcover,– 10 Oct 2006",1.369725,-0.342651,"For seven decades, ""Life"" has been thrilling t...",Photography Textbooks,"Arts, Film & Photography",965.62,42,-0.469338,42,659,1,1
5,ChiRunning: A Revolutionary Approach to Effort...,Danny Dreyer,"Paperback,– 5 May 2009",0.050347,-0.259449,The revised edition of the bestselling ChiRunn...,Healthy Living & Wellness (Books),Sports,900.00,71,-0.013062,71,1342,2,1
6,Death on the Nile (Poirot),Agatha Christie,"Paperback,– 5 Oct 2017",0.050347,0.501255,Agatha Christie’s most exotic murder mystery\n...,"Crime, Thriller & Mystery (Books)","Crime, Thriller & Mystery",224.00,26,0.025237,26,480,69,1
7,Yoga Your Home Practice Companion: A Complete ...,Sivananda Yoga Vedanta Centre,"Hardcover,– Import, 1 Mar 2018",0.050347,-0.164361,"Achieve a healthy body, mental alertness, and ...",Sports Training & Coaching (Books),Sports,836.00,169,-0.008275,169,691,2,1
8,Karmayogi: A Biography of E. Sreedharan,M S Ashokan,"Paperback,– 15 Dec 2015",0.050347,0.964810,Karmayogi is the dramatic and inspiring story ...,Biographies & Autobiographies (Books),"Biographies, Diaries & True Accounts",130.00,39,0.048575,39,864,1,1
9,"The Iron King (The Accursed Kings, Book 1)",Maurice Druon,"Paperback,– 26 Mar 2013",0.050347,-0.342651,‘This is the original game of thrones’ George ...,Action & Adventure (Books),Action & Adventure,695.00,42,-0.017251,42,926,3,-1


In [27]:
author_target_mean = df.groupby('Author')['Price'].mean()
df['Author encoded'] = df['Author'].map(author_target_mean)
df.drop(columns=['Author'], inplace=True)   

In [28]:
df.columns

Index(['Title', 'Edition', 'Reviews', 'Ratings', 'Synopsis', 'Genre',
       'BookCategory', 'Price', 'Title len', 'Rating Review score',
       'Title Char length', 'Synopsis len', 'Author freq', 'Synopsis Polarity',
       'Author encoded'],
      dtype='object')

In [ ]:
df.drop(["Title","BookCategory"], axis=1, inplace=True)


In [31]:
df["Genre Freq"] = df["Genre"].map(df["Genre"].value_counts())

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Limit max_features to avoid high dimensionality (adjust as needed)
tfidf = TfidfVectorizer(max_features=300, stop_words='english')

# Fit and transform the Synopsis column
tfidf_matrix = tfidf.fit_transform(df['Synopsis'].fillna(""))

# Convert to DataFrame and rename columns
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f"tfidf_syn_{word}" for word in tfidf.get_feature_names_out()])

# Concatenate with original dataframe
df = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)


In [34]:
df.drop("Synopsis",axis=1,inplace=True)
# One-hot encode the 'Genre' column
genre_dummies = pd.get_dummies(df['Genre'], prefix='Genre',dtype=int)

# Concatenate the new one-hot columns with the original DataFrame
df = pd.concat([df.drop('Genre', axis=1), genre_dummies], axis=1)


In [36]:
df.drop("Edition",axis=1,inplace=True)


In [37]:
df

,Reviews,Ratings,Price,Title len,Rating Review score,Title Char length,Synopsis len,Author freq,Synopsis Polarity,Author encoded,...,Genre_Vocabulary Books,"Genre_Walking, Hiking & Trekking (Books)",Genre_Waste Management,"Genre_Words, Language & Grammar Reference",Genre_Workbooks,Genre_World African & Middle Eastern Literature,Genre_Writing Guides (Books),Genre_XHTML Software Programming,Genre_Young Adults' Money & Jobs (Books),Genre_Zoology
0,0.050347,-0.259449,220.00,35,-0.013062,35,791,4,1,227.00000,...,0,0,0,0,0,0,0,0,0,0
1,-1.269032,-0.188133,202.93,34,0.238747,34,1146,1,1,202.93000,...,0,0,0,0,0,0,0,0,0,0
2,0.050347,-0.283221,299.00,28,-0.014259,28,1662,3,1,299.00000,...,0,0,0,0,0,0,0,0,0,0
3,0.050347,-0.200019,180.00,34,-0.010070,34,426,69,-1,241.17087,...,0,0,0,0,0,0,0,0,0,0
4,1.369725,-0.342651,965.62,42,-0.469338,42,659,1,1,965.62000,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6232,1.369725,-0.330765,322.00,50,-0.453058,50,1348,1,1,322.00000,...,0,0,0,0,0,0,0,0,0,0
6233,-1.269032,-0.247563,421.00,11,0.314166,11,1349,1,-1,421.00000,...,0,0,0,0,0,0,0,0,0,0
6234,-1.269032,-0.318879,399.00,54,0.404668,54,1285,3,1,449.00000,...,0,0,0,0,0,0,0,0,0,0
6235,-1.269032,-0.306993,319.00,28,0.389584,28,926,2,1,321.58000,...,0,0,0,0,0,0,0,0,0,0
